# Transaction Foundation Model — pretraining to serving on Ray

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: 45 min

### Anyscale Technical Demo — Ray Data + Ray Train + Ray Serve

---

## Business Context

Banks and fintechs are converging on **transaction foundation models** (TFMs): a single self-supervised transformer pretrained on raw transaction sequences, producing a reusable **customer embedding** that powers fraud, churn, credit, and personalization — replacing dozens of hand-built feature pipelines. Stripe, Visa (TREASURE), Nubank, and Revolut (PRAGMA) have all published variants of this recipe.

The model itself is small and not the hard part. The hard parts are **engineering at scale**: tokenizing petabytes of transactions, pretraining across many GPUs, and re-embedding every customer on a schedule — then serving those embeddings both in batch and in real time.

**This template** builds the whole pipeline on Ray, with one upgrade over the standard NVIDIA blueprint: a **static/dynamic field split** in the tokenizer and model (the idea behind Visa TREASURE and FATA-Trans), which is cheaper and a stronger inductive bias than flattening every field into the token stream.

---

## Architecture

```
                       ┌─────────────────────── Anyscale ───────────────────────┐
 raw transactions ──►  │ [Ray Data]   static/dynamic tokenization (map_groups)   │
   (Parquet, S3)       │ [Ray Train]  masked-feature pretraining (PyTorch + DDP)│
                       │ [Ray Data]   batch embedding extraction (CPU read+GPU)  │
                       │ [XGBoost]    downstream fraud: raw vs FM vs fusion       │
                       │ [Ray Serve]  online embedding + fraud score (cached)     │
                       └─────────────────────────────────────────────────────────┘
```

Every stage is the **same code** from laptop to multi-node cluster — you change `ScalingConfig`, not your program.

---

**Walkthrough:** this notebook runs end-to-end at `smoke` scale (a few thousand cards, a 2-layer model) so it completes in minutes on a small cluster. Flip `SCALE` to `small`/`full` and `USE_GPU=True` for the real distributed story.

## Get the code

```bash
git clone https://github.com/anyscale/templates && cd templates/templates/fintech_transaction_fm
```

## Step 1: Connect to the Ray Cluster

In an Anyscale Workspace, Ray is **pre-initialized** — no cluster setup, no Spark context. The install cell pulls the template's dependencies; on the GPU base image PyTorch is already present.

> In production you'd install from the generated `python_depset.lock`. Here we install from `requirements.txt` for portability.

In [1]:
!pip install -q -r requirements.txt

Successfully registered `ray, torch` and 6 other packages to be installed on all cluster nodes.
View and update dependencies here: https://console.anyscale.com/cld_g54aiirwj1s8t9ktgzikqur41k/prj_f1j47h9srml4cyg962id75ms2e/workspaces/expwrk_78mtwtucrd61tjxf851krarzwr?workspace-tab=dependencies


In [2]:
import sys, os
DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.utils import print_cluster_resources
print_cluster_resources()

2026-06-22 17:53:39,905	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 10.0.57.126:6379...
2026-06-22 17:53:39,938	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at https://session-nridlis6nyy2hi9qv4il5lcj6q.i.anyscaleuserdata.com 
2026-06-22 17:53:39,954	INFO packaging.py:691 -- Creating a file package for local module '/home/ray/default_cld_g54aiirwj1s8t9ktgzikqur41k/templates/templates/fintech_transaction_fm'.
2026-06-22 17:53:39,964	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_03eaed61e4fa56ae.zip' (0.32MiB) to Ray cluster...
2026-06-22 17:53:39,966	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_03eaed61e4fa56ae.zip'.


Ray cluster resources:
  CPU                  32.0
  GPU                  1.0
  accelerator_type:T4  1.0
  anyscale/accelerator_shape:1xT4 1.0
  anyscale/cpu_only:true 3.0
  anyscale/node-group:16CPU-64GB 1.0
  anyscale/node-group:1xT4:8CPU-32GB 1.0
  anyscale/node-group:8CPU-32GB 1.0
  anyscale/node-group:head 1.0
  anyscale/provider:aws 4.0
  anyscale/region:us-west-2 4.0
  memory               171798691840.0
  object_store_memory  48109983743.0

Cluster nodes: 32
  10.0.57.126          alive=True  anyscale/node-group:head=1.0, object_store_memory=9603900211.0, anyscale/cpu_only:true=1.0, memory=34359738368.0, anyscale/provider:aws=1.0, anyscale/region:us-west-2=1.0
  10.0.49.22           alive=True  anyscale/accelerator_shape:1xT4=1.0, object_store_memory=9614106624.0, accelerator_type:T4=1.0, GPU=1.0, anyscale/node-group:1xT4:8CPU-32GB=1.0, memory=34359738368.0, anyscale/provider:aws=1.0, anyscale/region:us-west-2=1.0, CPU=8.0
  10.0.8.232           alive=True  object_store_memory=

/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## Step 2: Load & inspect transaction data

By default we use the **real IBM TabFormer dataset** (Padhi et al., ICASSP 2021 — the public benchmark for transaction foundation models: 24.4M transactions, ~6.1k cards, 0.12% fraud), downloaded once (~266MB) and sampled down to the scale's card budget. Each *card* has **static** fields (issuer, card type, BIN region, home state — derived where TabFormer lacks them) plus a time-ordered stream of transactions with **dynamic** fields (amount, merchant, MCC, hour, day-of-week) and a fraud label.

The loader also writes `splits.json` with **temporal 80/10/10 cutoffs** (train on the past, test on the most recent 10%) — the same evaluation protocol as NVIDIA's transaction-FM blueprint, so downstream numbers are comparable.

> Fully offline alternative: `python scripts/01_generate_data.py --source synthetic` generates schema-identical synthetic data.


In [3]:
import pandas as pd
from src.paths import SCALE_MAP, artifact_paths, get_demo_base_dir
from src.tabformer import prepare_tabformer

SCALE = "small"        # "small" / "full" for the distributed story
USE_GPU = True        # set True on a GPU cluster for train + embed

BASE_DIR = get_demo_base_dir()
paths = artifact_paths(BASE_DIR, SCALE)

if not (os.path.exists(paths["raw"]) and os.path.exists(paths["splits"])):
    prepare_tabformer(
        paths["raw"], paths["splits"],
        num_cards=SCALE_MAP[SCALE], source_dir=paths["source"],
    )

df = pd.read_parquet(paths["raw"])
print(f"{len(df):,} transactions / {df['card_id'].nunique():,} cards / fraud {df['is_fraud'].mean()*100:.3f}%")
df.head(4)

24,386,900 transactions / 6,139 cards / fraud 0.122%


,card_id,timestamp,amount,merchant_id,merchant_category,mcc,hour,day_of_week,is_fraud,issuer,bin_region,card_type,home_state
0,0,2002-09-01 06:21:00,134.09,3527213246127876953,retail,5300,6,6,0,UNKNOWN,UNKNOWN,swipe,CA
1,0,2002-09-01 06:42:00,38.48,-727612092139916043,grocery,5411,6,6,0,UNKNOWN,UNKNOWN,swipe,CA
2,0,2002-09-02 06:22:00,120.34,-727612092139916043,grocery,5411,6,0,0,UNKNOWN,UNKNOWN,swipe,CA
3,0,2002-09-02 17:45:00,128.95,3414527459579106770,retail,5651,17,0,0,UNKNOWN,UNKNOWN,swipe,CA


## Step 3: Tokenize with Ray Data — the static/dynamic split

This is the core idea. NVIDIA's blueprint flattens every transaction into ~12 tokens in one shared vocabulary, so a sequence of *N* transactions costs ~12*N* positions. We instead:

- embed **static** card-level fields **once** and broadcast them to every position (they never spend sequence length), and
- give each **dynamic** field its own embedding table, so each transaction is **one** position whose vector is the sum of its field embeddings.

The vocabulary is fully deterministic (fixed amount buckets + merchant hashing), so tokenization is a **stateless `map_groups`** with no global shuffle — exactly what Ray Data is built for. NVIDIA's RAPIDS path is single-GPU; this scales across the cluster.

Two representation choices live here. **Positions are time-aware**: alongside the ordinal position we embed the log-bucketed *gap since the previous transaction*, because for transactions *when* matters more than ordinal slot. And **amount uses bucketing** (`AMOUNT_MODE`), with an optional `"soft"` mode that blends the two adjacent bin embeddings so $86.99 and $87.01 don't land on unrelated vectors.

> **What you'll see while this runs** (and why it's fine):
> - **"Cluster does not have any available CPUs / job may hang"** — on a fresh or idle cluster the GPU worker is still launching; Ray Data waits and the warnings clear once it lands (~2 min). The head node intentionally schedules no work.
> - **Hash-shuffle aggregator warnings** — Ray's shuffle defaults assume a much bigger cluster; we right-size it per scale (`shuffle_partitions` + `max_hash_shuffle_aggregators` in `scripts/02_tokenize.py`).
> - **"Numba isn't available"** — `numba` (in `requirements.txt`) lets RayTurbo JIT the hash-partitioning kernel of this groupby; without it the shuffle falls back to slower Python.
> - **"Object store is configured to use only 28% of memory"** — set at cluster start on Anyscale; at this dataset size (~3GB) the default is plenty.


In [4]:
import json

import numpy as np
from ray.data.expressions import col
from src.tokenizer import SEQ_LEN_BY_SCALE, tokenize_dataset, write_vocab

seq_len = SEQ_LEN_BY_SCALE[SCALE]
with open(paths["splits"]) as f:
    splits = json.load(f)

ds = ray.data.read_parquet(paths["raw"])
tokenized = tokenize_dataset(
    ds, seq_len,
    train_end=splits["train_end"],   # temporal cutoffs -> pretrain never sees val/test
    val_end=splits["val_end"],
    normal_keep=0.005,               # downsample normal eval samples (all frauds kept)
    max_pretrain_windows=8,
    num_partitions=32,            # right-size the shuffle (Ray defaults to 200)
).materialize()

PRETRAIN_DROP = ["kind", "split", "label", "weight",
                 "raw_amount", "raw_hour", "raw_dow", "raw_mcc", "raw_ts"]
tokenized.filter(expr=col("kind") == "pretrain").drop_columns(PRETRAIN_DROP) \
    .write_parquet(paths["tokenized_pretrain"])
tokenized.filter(expr=col("kind") == "eval").drop_columns(["kind"]) \
    .write_parquet(paths["tokenized_eval"])
write_vocab(paths["vocab"])

tok = pd.read_parquet(paths["tokenized_eval"])
print(f"{len(tok):,} eval samples ({int((tok['label'] == 1).sum()):,} fraud), seq_len={seq_len}")
row = tok.iloc[0]
print("  static:", {c: int(row[c]) for c in tok.columns if c.startswith("s_")})
print("  amount-bucket tokens:", np.asarray(row["d_amount_bucket"])[-8:].tolist())
print("  attention mask:", np.asarray(row["attention_mask"])[-8:].tolist())

2026-06-22 17:53:45,858	INFO logging.py:416 -- Registered dataset logger for dataset dataset_79_0
2026-06-22 17:53:45,883	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_79_0. Full logs are in /tmp/ray/session_2026-06-22_16-24-31_081523_2710/logs/ray-data
2026-06-22 17:53:45,884	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_79_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> HashShuffleOperator[Shuffle(key_columns=('card_id',), num_partitions=32)] -> TaskPoolMapOperator[MapBatches(tokenize_group)]
2026-06-22 17:53:45,917	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 28.0% of available memory (44.8GiB out of 160.0GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STO

(autoscaler +18s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.
(autoscaler +18s) [autoscaler] [48CPU-192GB] Attempting to add 1 node to the cluster (increasing from 0 to 1).


2026-06-22 17:54:03,709	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_79_0 execution finished in 17.82 seconds
INFO:openlineage.client.client:OpenLineageClient will use `composite` transport
INFO:openlineage.client.transport.composite:Stopping OpenLineage CompositeTransport emission after the first successful delivery because `continue_on_success=False`. Transport that emitted the event: <HttpTransport(name=first, kind=http, priority=1)>
2026-06-22 17:54:04,024	INFO logging.py:416 -- Registered dataset logger for dataset dataset_84_0
2026-06-22 17:54:04,028	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_84_0. Full logs are in /tmp/ray/session_2026-06-22_16-24-31_081523_2710/logs/ray-data
2026-06-22 17:54:04,029	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_84_0: InputDataBuffer[Input] -> TaskPoolMapOperator[Filter(col('kind') == 'pretrain')->MapBatches(drop_columns)->Write]
2026-06-22 17:54:04,068	INFO logging_progress.py:174 -- ==

(autoscaler +25s) [autoscaler] [48CPU-192GB|m5.12xlarge] [us-west-2a] [on-demand] Launched 1 instance.


2026-06-22 17:54:08,288	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_89_0 execution finished in 3.09 seconds
INFO:openlineage.client.client:OpenLineageClient will use `composite` transport
INFO:openlineage.client.transport.composite:Stopping OpenLineage CompositeTransport emission after the first successful delivery because `continue_on_success=False`. Transport that emitted the event: <HttpTransport(name=first, kind=http, priority=1)>
2026-06-22 17:54:08,323	INFO dataset.py:5765 -- Data sink Parquet finished. 152103 rows and 619.4MiB data written.


912,618 eval samples (178,542 fraud), seq_len=128
  static: {'s_issuer': 15, 's_card_type': 7, 's_bin_region': 9, 's_home_state': 7}
  amount-bucket tokens: [8, 9, 7, 10, 9, 8, 8, 7]
  attention mask: [1, 1, 1, 1, 1, 1, 1, 1]


## Step 4: Pretrain with Ray Train (masked-feature modeling, DDP)

We pretrain by **masking dynamic-field tokens and predicting them** (the Stripe / Open-Banking objective — bidirectional context beats next-token for the fixed-window tasks fintech cares about). There's **one classification head per dynamic field**, and because those heads differ wildly in difficulty (merchant is ~2000-way, day-of-week is 9-way) we balance them with **uncertainty weighting** (Kendall & Gal) so the big head doesn't dominate.

The training loop is plain PyTorch; **Ray Train** handles worker setup, dataset sharding, **DDP** wrapping, checkpointing, and fault tolerance. The model fits one GPU at every scale, so we use DDP (data-parallel) and scale by adding workers — go from 1 CPU worker (here) to N GPU workers by changing only `num_workers` and `use_gpu`. (`use_fsdp` is available for when the model itself outgrows a GPU.)

In [5]:
from src.pretrain import pretrain

metrics = pretrain(
    tokenized_path=paths["tokenized_pretrain"],
    vocab_path=paths["vocab"],
    checkpoint_out=paths["checkpoint"],
    size=SCALE,
    max_len=seq_len,
    epochs=2,
    batch_size=64,
    num_workers=1,          # bump to 4-8 GPU workers at scale
    use_gpu=USE_GPU,
    use_fsdp=False,         # True for sharded multi-GPU training
    storage_base=BASE_DIR,  # shared storage — workers run on other nodes
)
print("final MLM loss:", round(metrics["mlm_loss"], 4))

(TrainController pid=41845) A run snapshot was found in storage folder at: '/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain'
(TrainController pid=41845) This snapshot contains a list of checkpoints reported via `ray.train.report` and will be loaded. This allows the latest checkpoint found in the snapshot to be accessible within your training function via `ray.train.get_checkpoint`.
(TrainController pid=41845) If you meant to start a brand new training job without any information about previous checkpoints found in this directory, please configure a new, unique `RunConfig(name)` or delete the existing folder at '/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain'.
(TrainController pid=41845) Requesting resources: {'GPU': 1} * 1
(TrainController pid=41845) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=41845) Attempting to start training worker group of size 1 with the following resources: [{'GPU': 1}] * 1
(RayTrainWorker 

(pid=42130) Running Dataset train_92_0.: 0.00 row [00:00, ? row/s]

(pid=42130) - ListFiles:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=42130) - ReadFiles:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=42130) - split(1, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

...

(pid=42130) - ReadFiles:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=42130) - split(1, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(autoscaler +5m33s) [autoscaler] Cluster upscaled to {160 CPU, 1 GPU}.
(autoscaler +9m13s) [autoscaler] Cluster resized to {128 CPU, 1 GPU}.


(SplitCoordinator pid=42130) ✔️  Dataset train_92_1 execution finished in 248.76 seconds
(SplitCoordinator pid=42130) INFO:openlineage.client.client:OpenLineageClient will use `composite` transport
(SplitCoordinator pid=42130) INFO:openlineage.client.transport.composite:Stopping OpenLineage CompositeTransport emission after the first successful delivery because `continue_on_success=False`. Transport that emitted the event: <HttpTransport(name=first, kind=http, priority=1)>
(RayTrainWorker pid=7539, ip=10.0.49.22) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain/checkpoint_2026-06-22_18-03-20.243218)
(RayTrainWorker pid=7539, ip=10.0.49.22) Reporting training result 8: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain/checkpoint_2026-06-22_18-03-20.243218), metrics={'epoch': 1, 'mlm_loss': 8.125502793468428, 'acc_a

[pretrain] final mlm_loss=8.1255 macro_acc=0.292 -> /mnt/cluster_storage/transaction-fm/model/small/
final MLM loss: 8.1255


## Step 5: Batch embedding extraction with Ray Data

The recurring production job: score every customer to a fresh embedding. This is a heterogeneous **CPU-read + GPU-infer** workload that streams through one Ray Data pipeline — the model loads once per replica, batches stream through, output is written idempotently. This is the stage with no clean public reference, and where Ray clearly earns its keep.

In [6]:
from src.embed import extract_embeddings

extract_embeddings(
    tokenized_path=paths["tokenized_eval"],
    checkpoint_dir=paths["checkpoint"],
    output_path=paths["embeddings"],
    num_workers=1,          # scale out across GPU replicas at `full`
    use_gpu=USE_GPU,
)
emb = pd.read_parquet(paths["embeddings"])
dim = np.asarray(emb["embedding"].iloc[0]).shape[0]
print(f"{len(emb):,} transaction-window embeddings, dim={dim}")

2026-06-22 18:03:23,958	INFO logging.py:416 -- Registered dataset logger for dataset dataset_97_0
2026-06-22 18:03:23,963	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_97_0. Full logs are in /tmp/ray/session_2026-06-22_16-24-31_081523_2710/logs/ray-data
2026-06-22 18:03:23,964	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_97_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[MapBatches(EmbeddingExtractor)] -> TaskPoolMapOperator[Write]
{"asctime":"2026-06-22 18:03:23,988","levelname":"E","message":"Actor with class name: 'MapWorker(MapBatches(EmbeddingExtractor))' and ID: '9ba290edc926f70e39e51e6f06000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost, the actor restart will fail. See https://github.com/ray-project/ray/issues/53727 for more details.","filename":"core_worker.cc","lineno":

(MapWorker(MapBatches(EmbeddingExtractor)) pid=9450, ip=10.0.49.22) [embed] loaded TransactionFM on cuda


(MapWorker(MapBatches(EmbeddingExtractor)) pid=9450, ip=10.0.49.22) /tmp/ray/session_2026-06-22_16-24-31_081523_2710/runtime_resources/working_dir_files/_ray_pkg_03eaed61e4fa56ae/src/embed.py:56: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
(MapWorker(MapBatches(EmbeddingExtractor)) pid=9450, ip=10.0.49.22)   return torch.as_tensor(v, dtype=dtype, device=self.device)
2026-06-22 18:03:34,175	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_97_0 =======
2026-06-22 18:03:34,176	INFO logging_progress.py:225 -- Total Progress: 3/192
2026-06-22 18:03:34,177	INFO logging_progress.py:227 -- Active & requested

(autoscaler +15m13s) [autoscaler] Cluster resized to {96 CPU, 1 GPU}.


2026-06-22 18:08:55,500	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_97_0 =======
2026-06-22 18:08:55,501	INFO logging_progress.py:225 -- Total Progress: 191/192
2026-06-22 18:08:55,501	INFO logging_progress.py:227 -- Active & requested resources: 0/128 CPU, 1/1 GPU, 110.6MiB/72.5GiB object store
2026-06-22 18:08:55,502	INFO logging_progress.py:181 -- 
2026-06-22 18:08:55,502	INFO logging_progress.py:231 -- ListFiles: 192/192
2026-06-22 18:08:55,503	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store
2026-06-22 18:08:55,503	INFO logging_progress.py:231 -- ReadFiles: 912618/912618
2026-06-22 18:08:55,504	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 105.6MiB object store
2026-06-22 18:08:55,505	INFO logging_progress.py:231 -- MapBatches(EmbeddingExtractor): 905079/909818
2026-06-22 18:08:55,505	INFO logging_progress.py:233 --   Tasks: 1; Actors: 1; 

[embed] wrote customer embeddings -> /mnt/cluster_storage/transaction-fm/embeddings/small/


2026-06-22 18:09:00,211	INFO logging.py:416 -- Registered dataset logger for dataset dataset_100_0
2026-06-22 18:09:00,214	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_100_0. Full logs are in /tmp/ray/session_2026-06-22_16-24-31_081523_2710/logs/ray-data
2026-06-22 18:09:00,215	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_100_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> LimitOperator[limit=2000]
2026-06-22 18:09:00,231	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_100_0 =======
2026-06-22 18:09:00,232	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-06-22 18:09:00,232	INFO logging_progress.py:227 -- Active & requested resources: 0/128 CPU, 0.0B/72.5GiB object store
2026-06-22 18:09:00,233	INFO logging_progress.py:181 -- 
2026-06-22 18:09:00,234	INFO logging_progress.py:231 -- ListFiles: 0/1
2026-06-22 18:09:00,235	INFO logging_progress.py:233 --   Tasks: 1; A

[embed] health: mean_pairwise_cos=0.096 (→1 = collapse), mean_feature_var=3.2234, n=2000
(autoscaler +15m28s) [autoscaler] Cluster upscaled to {128 CPU, 1 GPU}.
2,585,751 transaction-window embeddings, dim=256


## Step 6: Downstream fraud — raw vs FM vs fusion

The headline result, evaluated with the **NVIDIA transaction-FM blueprint protocol**: temporal 80/10/10 split, per-transaction last-event fraud labels, AUC-ROC + PR-AUC at natural fraud prevalence (downsampled normals are importance-weighted back). Same XGBoost recipe, three feature sets:

1. **raw** — the target transaction's tabular fields (what you have today)
2. **fm** — the FM embedding of the history window only
3. **fusion** — embedding ++ raw features (Nubank's joint fusion)

The lift of (2) and (3) over (1) is the case for a transaction FM. *(At `smoke` scale — 2 CPU epochs, a 2-layer model — expect fusion ≈ raw; the gap opens with the `small`/`full` GPU pretrain.)*


In [9]:
from src.downstream import run_downstream, print_summary

summary = run_downstream(paths["embeddings"], paths["downstream"])
print_summary(summary)

[05] per-sample test scores -> /mnt/cluster_storage/transaction-fm/downstream/small//test_predictions.parquet
protocol: temporal 80/10/10 split by transaction time; per-transaction last-event fraud labels; prevalence-weighted metrics on the held-out most-recent 10% (NVIDIA transaction-FM blueprint protocol)
samples  train=2,077,757  val=241,366  test=266,628 (natural test fraud rate 0.1047%)
eval fingerprint: a52f3ef32b9b4972 (metrics comparable across runs iff this matches)
feature set    AUC-ROC     PR-AUC
--------------------------------
raw            0.8495     0.0225
fm             0.8215     0.0082
fusion         0.8613     0.0156
--------------------------------
FM-only PR-AUC lift vs raw:  -0.0143
Fusion PR-AUC lift vs raw:   -0.0069


## Step 7: Online serving with Ray Serve

The default production path is batch (Step 5) → feature store → XGBoost. But fraud also needs a real-time path, so we ship a **Ray Serve** deployment that mirrors the two-tier pattern real shops use: it **caches static (card-level) embeddings** and runs the transformer only over the recent dynamic window, returning an embedding + fraud score in one call. Autoscales 1→4 replicas.

In [10]:
import requests
from ray import serve
from src.serve import build_app
from src.utils import sample_serve_payload

serve.run(build_app(paths["checkpoint"]), name="txn-fm")
payload = sample_serve_payload(paths["tokenized_eval"])
resp = requests.post("http://localhost:8000/", json=payload, timeout=30).json()
print("card_id   :", resp["card_id"])
print("embedding :", [round(x, 3) for x in resp["embedding"][:6]], "...")
print("fraud_score:", round(resp["fraud_score"], 4))
serve.shutdown()

(ProxyActor pid=57492) INFO 2026-06-22 18:33:11,924 proxy 10.0.57.126 -- Proxy starting on node 176c50d3fa5fc21423081844295555a4967fabdb375d85ead889da58 (HTTP port: 8000).
INFO 2026-06-22 18:33:12,114 serve 41232 -- Started Serve in namespace "serve".
(ServeController pid=57423) INFO 2026-06-22 18:33:12,153 controller 57423 -- Registering autoscaling state for deployment Deployment(name='TransactionEmbeddingService', app='txn-fm')
(ServeController pid=57423) INFO 2026-06-22 18:33:12,154 controller 57423 -- Deploying new version of Deployment(name='TransactionEmbeddingService', app='txn-fm') (initial target replicas: 1).
(ProxyActor pid=57492) INFO 2026-06-22 18:33:12,110 proxy 10.0.57.126 -- Got updated endpoints: {}.
(ProxyActor pid=57492) INFO 2026-06-22 18:33:12,158 proxy 10.0.57.126 -- Got updated endpoints: {Deployment(name='TransactionEmbeddingService', app='txn-fm'): EndpointInfo(route='/', app_is_cross_language=False, route_patterns=None)}.
(ProxyActor pid=57492) INFO 2026-06-2

card_id   : 203
embedding : [-0.985, 0.716, -0.172, -0.718, -1.773, 0.32] ...
fraud_score: 0.2173


(ServeController pid=57423) INFO 2026-06-22 18:33:21,282 controller 57423 -- Replica(id='m573dm0l', deployment='TransactionEmbeddingService', app='txn-fm') is stopped.
(ServeController pid=57423) INFO 2026-06-22 18:33:21,282 controller 57423 -- Deregistering autoscaling state for deployment Deployment(name='TransactionEmbeddingService', app='txn-fm')
(ServeController pid=57423) INFO 2026-06-22 18:33:21,283 controller 57423 -- Deregistering autoscaling state for application txn-fm


(raylet) Task ServeController.graceful_shutdown failed. There are infinite retries remaining, so the task will be retried. Error: The actor is dead because it was killed by `ray.kill`.


## Step 8: Observability & fault tolerance

**Observability** (built into Anyscale)
- **Ray Dashboard** — watch data stream through tokenize/embed stages, per-worker throughput, GPU utilization.
- **Ray Train reports** — per-epoch MLM loss, checkpoint lineage.
- **Serve metrics** — latency, replica autoscaling, ongoing requests.

**Fault tolerance** (built into Ray)
- Ray Data retries failed batches per-partition — no full restart.
- Ray Train checkpoints let pretraining resume after a node loss.
- Streaming + backpressure keeps memory bounded across stages.

## Step 9: Path to production

The same code scales up by changing config, and runs as a scheduled **Anyscale Job**. `scripts/run_pipeline.py` wraps all six stages (data -> tokenize -> pretrain -> embed -> downstream -> validate) in one command:

```bash
# Full pipeline as a Job (GPU workers, autoscaling):
anyscale job submit --config-file job_config.yaml

# Larger scale:
anyscale job submit --config-file job_config.yaml -- python scripts/run_pipeline.py --scale full
```

| Stage | Ray primitive | Scale knob |
|-------|---------------|-----------|
| Tokenize | Ray Data `map_groups` | partitions / cluster size |
| Pretrain | Ray Train + DDP | `num_workers`, `use_gpu` |
| Batch embed | Ray Data `map_batches` | `num_workers`, `num_gpus` |
| Online serve | Ray Serve | replica autoscaling |


## Validate

A quick end-to-end check that every stage produced sane artifacts.

In [11]:
from scripts.validate_results import validate_pipeline, print_report
print_report(validate_pipeline(paths))

Pipeline validation:
  pretrain wins:  208,026
  sequences:      2,585,751
  embedding dim:  256
  embeddings:     2,585,751
  raw     PR-AUC=0.0225  AUC-ROC=0.8495
  fm      PR-AUC=0.0082  AUC-ROC=0.8215
  fusion  PR-AUC=0.0156  AUC-ROC=0.8613
  fusion lift:    -0.0069 PR-AUC vs raw
  ALL CHECKS PASSED
